# VLM-Anomaly — Full MVTec Sweep · Gemini 2.5 Flash (Local)
**Runs locally against your `.env` API key and `data/mvtec` dataset.**

In [9]:
# ── Cell 1: Setup paths & sys.path ─────────────────────────────────────────
import sys, os
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT = Path().resolve().parent  # notebooks/ → repo root
SRC_DIR   = REPO_ROOT / 'src'
PROMPTS_DIR = REPO_ROOT / 'prompts'
RESULTS_DIR = REPO_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert SRC_DIR.exists(), f'src/ not found at {SRC_DIR}'
assert PROMPTS_DIR.exists(), f'prompts/ not found at {PROMPTS_DIR}'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

load_dotenv(REPO_ROOT / '.env')

import vlm_anomaly
print(f"vlm_anomaly {vlm_anomaly.__version__} ready")
print(f"SRC      : {SRC_DIR}")
print(f"Prompts  : {PROMPTS_DIR}")
print(f"Results  : {RESULTS_DIR}")

vlm_anomaly 0.1.0 ready
SRC      : /Users/sabareeswarans/Projects_26/VLM-Anomaly/src
Prompts  : /Users/sabareeswarans/Projects_26/VLM-Anomaly/prompts
Results  : /Users/sabareeswarans/Projects_26/VLM-Anomaly/results


In [10]:
# ── Cell 2: Verify API key ──────────────────────────────────────────────────
import os

api_key = os.environ.get('GEMINI_API_KEY', '')
assert api_key, 'GEMINI_API_KEY not set — add it to your .env file'
print(f"GEMINI_API_KEY: {api_key[:8]}...{api_key[-4:]}")

GEMINI_API_KEY: AIzaSyA1...xi2k


In [11]:
# ── Cell 3: Find MVTec dataset ──────────────────────────────────────────────
MVTEC_ROOT = None
for candidate in [
    REPO_ROOT / 'data' / 'mvtec',
    REPO_ROOT / 'data' / 'mvtec-ad',
    Path('/tmp/mvtec'),
]:
    if candidate.exists() and any(candidate.iterdir()):
        MVTEC_ROOT = candidate
        break

assert MVTEC_ROOT, f'MVTec not found. Expected at {REPO_ROOT}/data/mvtec'
categories = sorted([d.name for d in MVTEC_ROOT.iterdir() if d.is_dir()])
print(f"MVTec root : {MVTEC_ROOT}")
print(f"Categories : {len(categories)} → {categories}")

MVTec root : /Users/sabareeswarans/Projects_26/VLM-Anomaly/data/mvtec
Categories : 15 → ['bottle', 'cable', 'capsule', 'carpet', 'grid', 'hazelnut', 'leather', 'metal_nut', 'pill', 'screw', 'tile', 'toothbrush', 'transistor', 'wood', 'zipper']


In [12]:
# ── Cell 4: Configure ───────────────────────────────────────────────────────
PROMPT_KEY = "manufacturing.detailed"
LIMIT      = None        # None = all images; set e.g. 5 for a quick test
BUDGET_USD = 5.0
MODEL      = "gemini-2.5-flash"

print(f"Prompt : {PROMPT_KEY}")
print(f"Limit  : {LIMIT or 'all images'}")
print(f"Model  : {MODEL}")
print(f"Total  : ~{len(categories) * 83} images")

Prompt : manufacturing.detailed
Limit  : all images
Model  : gemini-2.5-flash
Total  : ~1245 images


In [5]:
# ── Cell 5: Build shared objects ────────────────────────────────────────────
from vlm_anomaly.config import Settings
from vlm_anomaly.datasets.mvtec import MVTec
from vlm_anomaly.backends.gemini import GeminiBackend
from vlm_anomaly.evaluators.prompt_library import PromptLibrary
from vlm_anomaly.logging import configure_logging

configure_logging(json_logs=False, log_level='INFO')

settings = Settings(
    _env_file=str(REPO_ROOT / '.env'),
    data_dir=str(MVTEC_ROOT.parent),
    results_dir=str(RESULTS_DIR),
    default_budget_usd=BUDGET_USD,
)
settings.results_dir = RESULTS_DIR

dataset    = MVTec(root_dir=MVTEC_ROOT)
backend    = GeminiBackend(model=MODEL)
prompt_lib = PromptLibrary(prompts_dir=PROMPTS_DIR)

print(f"Dataset  : {MVTEC_ROOT}")
print(f"Backend  : {backend.name} / {MODEL}")
print(f"Prompts  : {len(list(prompt_lib._prompts.keys()))} keys loaded")

Dataset  : /Users/sabareeswarans/Projects_26/VLM-Anomaly/data/mvtec
Backend  : gemini / gemini-2.5-flash
Prompts  : 4 keys loaded


In [ ]:
# ── Cell 6: Run all 15 categories ───────────────────────────────────────────
from tqdm.auto import tqdm
from vlm_anomaly.schemas import ExperimentConfig
from vlm_anomaly.evaluators.vlm_evaluator import VLMEvaluator

all_results = []

for category in tqdm(categories, desc='MVTec categories'):
    existing = [f for f in RESULTS_DIR.glob(f'*_mvtec_{category}.jsonl')
                if 'gemini' in f.name and f.stat().st_size > 100]
    if existing:
        print(f"  [skip] {category} — already done ({existing[0].name})")
        continue

    config = ExperimentConfig(
        backend=f'gemini/{MODEL}',
        dataset='mvtec',
        categories=[category],
        prompt=PROMPT_KEY,
        limit=LIMIT,
        budget_usd=BUDGET_USD,
    )
    evaluator = VLMEvaluator(
        backend=backend,
        dataset=dataset,
        config=config,
        settings=settings,
        prompt_library=prompt_lib,
    )
    results = evaluator.run()
    all_results.extend(results)
    for r in results:
        print(f"  {category:12s}  AUROC={r.auroc:.3f}  F1={r.f1:.3f}  n={r.n_images}")

print(f"\nDone. {len(all_results)} categories processed.")

In [ ]:
# ── Cell 6b: Run ONLY missing Gemini categories ────────────────────────────
# Checks which categories Gemini has already completed (by reading model_id
# from file content) and runs only the ones that are absent.
import json as _json

def _gemini_done_categories(results_dir, model_id='gemini/gemini-2.5-flash'):
    done = set()
    for f in results_dir.glob('*_mvtec_*.jsonl'):
        if f.stat().st_size < 100:
            continue
        try:
            rec = _json.loads(f.read_text().splitlines()[0])
            if rec.get('model_id') == model_id:
                done.add(rec.get('category', ''))
        except Exception:
            pass
    return done

done_cats  = _gemini_done_categories(RESULTS_DIR)
all_cats   = set(categories)
missing    = sorted(all_cats - done_cats)

print(f'Already done : {sorted(done_cats)}')
print(f'Missing      : {missing}')

if not missing:
    print('All 15 categories complete — nothing to run.')
else:
    missing_results = []
    for category in tqdm(missing, desc='Missing categories'):
        config = ExperimentConfig(
            backend=f'gemini/{MODEL}',
            dataset='mvtec',
            categories=[category],
            prompt=PROMPT_KEY,
            limit=LIMIT,
            budget_usd=BUDGET_USD,
        )
        evaluator = VLMEvaluator(
            backend=backend,
            dataset=dataset,
            config=config,
            settings=settings,
            prompt_library=prompt_lib,
        )
        results = evaluator.run()
        missing_results.extend(results)
        for r in results:
            print(f'  {category:12s}  AUROC={r.auroc:.3f}  F1={r.f1:.3f}  n={r.n_images}')
    print(f'\nDone. {len(missing_results)} missing categories filled in.')


In [8]:
# ── Cell 7: Leaderboard ─────────────────────────────────────────────────────
import importlib
import vlm_anomaly.analysis.aggregator as _agg_mod
importlib.reload(_agg_mod)
from vlm_anomaly.analysis.aggregator import leaderboard, cost_accuracy_table

lb = leaderboard(RESULTS_DIR)
if lb.empty:
    print('No results yet.')
else:
    summary = cost_accuracy_table(RESULTS_DIR)
    print('=== Summary (mean across categories) ===')
    print(summary[["model_id","mean_auroc","mean_latency_ms"]].to_string(index=False))
    print()
    print('=== Per-category breakdown (all models) ===')
    clean = lb[lb["model_id"].notna() & lb["category"].notna() & (lb["n_images"] > 10)]
    display(
        clean[["model_id","category","n_images","auroc","f1","mean_latency_ms"]]
        .sort_values(["model_id","auroc"], ascending=[True,False])
        .reset_index(drop=True)
    )

/Users/sabareeswarans/Projects_26/VLM-Anomaly/src/vlm_anomaly/analysis/aggregator.py:78: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(frames, ignore_index=True)


=== Summary (mean across categories) ===
                             model_id  mean_auroc  mean_latency_ms
              gemini/gemini-2.5-flash    0.621818      3967.419299
openrouter/qwen/qwen3-vl-32b-instruct    0.181144      2153.340848

=== Per-category breakdown (all models) ===


/Users/sabareeswarans/Projects_26/VLM-Anomaly/src/vlm_anomaly/analysis/aggregator.py:78: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(frames, ignore_index=True)


,model_id,category,n_images,auroc,f1,mean_latency_ms
0,gemini/gemini-2.5-flash,screw,160,0.796885,0.884120,4040.841105
1,gemini/gemini-2.5-flash,tile,117,0.786977,0.988095,3945.857379
2,gemini/gemini-2.5-flash,hazelnut,110,0.679286,0.833333,4008.899553
3,gemini/gemini-2.5-flash,grid,78,0.676692,0.973451,3929.244934
4,gemini/gemini-2.5-flash,metal_nut,115,0.639296,0.908163,3933.382661
5,gemini/gemini-2.5-flash,capsule,132,0.613682,0.888889,3997.731006
6,gemini/gemini-2.5-flash,wood,79,0.591667,0.929134,4071.416938
7,gemini/gemini-2.5-flash,transistor,100,0.591667,0.612903,4043.524536
8,gemini/gemini-2.5-flash,pill,167,0.554010,0.918033,3894.512138
9,gemini/gemini-2.5-flash,toothbrush,42,0.550000,0.848485,3984.961829


In [8]:
# ── Cell 8: Generate report ──────────────────────────────────────────────────
from vlm_anomaly.analysis.report_generator import generate

report = generate(RESULTS_DIR, str(REPO_ROOT / 'REPORT.md'))
print(f"Report written to {report}")

2026-05-23T21:31:48.541738Z [info     ] report.generate.start          [vlm_anomaly.analysis.report_generator] results_dir=/Users/sabareeswarans/Projects_26/VLM-Anomaly/results
/Users/sabareeswarans/Projects_26/VLM-Anomaly/src/vlm_anomaly/analysis/aggregator.py:78: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(frames, ignore_index=True)
/Users/sabareeswarans/Projects_26/VLM-Anomaly/src/vlm_anomaly/analysis/aggregator.py:78: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combi

Report written to /Users/sabareeswarans/Projects_26/VLM-Anomaly/REPORT.md
